# Mancala (Kalah) AI Engine
## Penn State AI 801 -- Group 8 (Matt Kolker, Elliott Armstrong, Shuyang Huang)

A complete Mancala game engine with five AI agents for comparative analysis:
**Random, Greedy, Minimax, Alpha-Beta, and MCTS.**

**Modes:**
1. **Single Game** -- Watch two agents play, or play against an AI
2. **Quick Matchup** -- Run N games between two agents with summary stats
3. **Tournament** -- Automated multi-scenario tournament with optional CSV output

**Features:**
- Two evaluation functions: standard (store difference) and rich (one-ply simulation)
- Configurable MCTS exploration constant (C parameter)
- Extra turns do not consume search depth in minimax/alpha-beta
- Cross-machine timing normalization via deterministic benchmark
- 40-column CSV output for statistical analysis

**Setup:** Run Cells 1--2 to load the engine, then use Cells 3+ for your chosen mode.

In [ ]:
# Cell 1: Load the engine from mancala.py
# Place mancala.py in the same directory as this notebook.
import os

ENGINE_PATH = 'mancala.py'  # <-- Update path if needed
if os.path.exists(ENGINE_PATH):
    with open(ENGINE_PATH) as f:
        engine_code = f.read().split("if __name__ == '__main__':")[0]
        exec(engine_code, globals())
    print(f'Engine loaded from {ENGINE_PATH}')
else:
    print(f'ERROR: {ENGINE_PATH} not found.')
    print('Place mancala.py in the notebook directory, or update ENGINE_PATH above.')

In [ ]:
# Cell 2: Agent catalog and benchmark
# The benchmark measures raw CPU speed for cross-machine timing normalization.
MACHINE_ID = 'notebook'
bm = run_benchmark(10)
print(f'Benchmark: {bm:.1f} ms (AB d=6 on initial state)')

# All available agents
# To add an agent, add an entry to AGENT_CATALOG in mancala.py and reload Cell 1.
# To test a different C parameter:
#   AGENT_CATALOG['MCTS(n=5000,C=0.7)'] = ('MCTS(n=5000,C=0.7)',
#       make_player(mcts_player, n_simulations=5000, C=0.7))

agent_names = list(AGENT_CATALOG.keys())
print(f'\nAvailable agents ({len(agent_names)}):')
for i, name in enumerate(agent_names):
    print(f'  {i+1:>2}. {name}')

## Mode 1: Single Game
Watch two agents play a single game with move-by-move output.
Change `PLAYER_1` and `PLAYER_2` below to select agents.

In [ ]:
# Single Game -- change these to pick agents
PLAYER_1 = 'Alpha-Beta(d=6)'   # <-- Change here
PLAYER_2 = 'MCTS(n=5000)'      # <-- Change here

p1_label, p1_fn = AGENT_CATALOG[PLAYER_1]
p2_label, p2_fn = AGENT_CATALOG[PLAYER_2]
print(f'{p1_label} (P1) vs {p2_label} (P2)\n')

game = Mancala()
state = game.initial
move_num = 0

while not game.terminal_test(state):
    current = state.to_move
    fn = p1_fn if current == 1 else p2_fn
    label = p1_label if current == 1 else p2_label

    t0 = time.perf_counter()
    move = fn(game, state)
    elapsed = time.perf_counter() - t0
    move_num += 1

    state = game.result(state, move)
    s1, s2 = state.board[P1_STORE], state.board[P2_STORE]
    print(f'  {move_num:>3}. {label:<25s} pit {move}  -> {s1:>2}-{s2:<2}  ({elapsed*1000:.0f}ms)')

game.display(state)
s1, s2 = game.score(state)
winner = 'P1' if s1 > s2 else ('P2' if s2 > s1 else 'Draw')
print(f'Result: {s1}-{s2} ({winner})')

## Mode 2: Quick Matchup
Run N games between two agents. Each agent plays as first mover and second mover
an equal number of times for balanced first-mover analysis.

Set `CSV_OUTPUT` to a filename to save game-level data, or `None` for screen only.

In [ ]:
# Quick Matchup -- configure here
PLAYER_A = 'MCTS(n=10000)'       # <-- Change agent
PLAYER_B = 'Alpha-Beta(d=6)'     # <-- Change agent
N_GAMES_PER_SEAT = 25            # Total games = 2x this
CSV_OUTPUT = None                 # Set to 'results.csv' to save, or None for screen only

a_label, a_fn = AGENT_CATALOG[PLAYER_A]
b_label, b_fn = AGENT_CATALOG[PLAYER_B]

summary = run_matchup(a_fn, b_fn, a_label, b_label,
                      n_games=N_GAMES_PER_SEAT,
                      csv_path=CSV_OUTPUT,
                      simulation_id=datetime.now().strftime('%Y%m%d_%H%M%S'),
                      scenario_name='Quick matchup')

print(f'\n{"=" * 60}')
print(f'  {summary["matchup_label"]}')
print(f'  {summary["p1_wins"]}W - {summary["p2_wins"]}L - {summary["draws"]}D')
print(f'  Win rate: {summary["p1_wins"]/summary["total_games"]*100:.1f}%')
print(f'  Avg score: {summary["p1_avg_score"]:.1f} - {summary["p2_avg_score"]:.1f}')
print(f'  Avg moves: {summary["avg_moves"]:.0f}')
print(f'  Time: P1={summary["p1_total_time"]:.1f}s, P2={summary["p2_total_time"]:.1f}s')
print(f'{"=" * 60}')
if CSV_OUTPUT:
    print(f'Results saved to: {CSV_OUTPUT}')

## Mode 3: Tournament
Run a full tournament across many agent configurations. The default tournament
covers calibration, depth scaling, pruning comparison, MCTS vs AB crossover,
MCTS budget scaling, self-play, and a random baseline (~28 matchups, ~2-3 hours).

### Customization
- Edit the matchup lists below to add or remove matchups
- Uncomment the EXTENDED sections for expensive high-N MCTS runs, eval function
  comparison, or C parameter sweeps
- Set `CSV_PATH` to save results, or `None` for screen-only display
- Adjust `GAMES_PER_SEAT` for more or fewer games (50 total per matchup at default)

In [ ]:
# Tournament configuration
CSV_PATH = None              # Set to 'mancala_results.csv' to save, or None for screen only
GAMES_PER_SEAT = 25          # Total per matchup = 2x this

# --- Helper functions for building matchups ---
def agent(name):
    return AGENT_CATALOG[name]

def matchup(a_name, b_name):
    a_label, a_fn = agent(a_name)
    b_label, b_fn = agent(b_name)
    return (a_label, a_fn, b_label, b_fn)

def self_play(name):
    label, fn = agent(name)
    return (label, fn, label, fn)

# ============================================================
# DEFINE TOURNAMENT MATCHUPS
# Each scenario: ('Name', [list of (p1_label, p1_fn, p2_label, p2_fn)])
# ============================================================

scenarios = [
    # --- 1. Calibration: every core agent vs Greedy ---
    ('Calibration vs Greedy', [
        self_play('Greedy'),
        matchup('Minimax(d=4)', 'Greedy'),
        matchup('Minimax(d=6)', 'Greedy'),
        matchup('Alpha-Beta(d=4)', 'Greedy'),
        matchup('Alpha-Beta(d=6)', 'Greedy'),
        matchup('Alpha-Beta(d=8)', 'Greedy'),
        matchup('MCTS(n=1000)', 'Greedy'),
        matchup('MCTS(n=5000)', 'Greedy'),
        matchup('MCTS(n=10000)', 'Greedy'),
    ]),

    # --- 2. AB depth scaling ---
    ('AB depth scaling', [
        matchup('Alpha-Beta(d=4)', 'Alpha-Beta(d=6)'),
        matchup('Alpha-Beta(d=6)', 'Alpha-Beta(d=8)'),
        matchup('Alpha-Beta(d=4)', 'Alpha-Beta(d=8)'),
    ]),

    # --- 3. Minimax vs AB at matched depth ---
    ('MM vs AB (same depth)', [
        matchup('Minimax(d=4)', 'Alpha-Beta(d=4)'),
        matchup('Minimax(d=6)', 'Alpha-Beta(d=6)'),
    ]),

    # --- 4. MCTS vs AB: the core comparison ---
    ('MCTS vs Alpha-Beta', [
        matchup('MCTS(n=1000)',  'Alpha-Beta(d=4)'),
        matchup('MCTS(n=5000)',  'Alpha-Beta(d=4)'),
        matchup('MCTS(n=10000)', 'Alpha-Beta(d=4)'),
        matchup('MCTS(n=5000)',  'Alpha-Beta(d=6)'),
        matchup('MCTS(n=10000)', 'Alpha-Beta(d=6)'),
        matchup('MCTS(n=10000)', 'Alpha-Beta(d=8)'),
    ]),

    # --- 5. MCTS budget scaling ---
    ('MCTS internal scaling', [
        matchup('MCTS(n=1000)',  'MCTS(n=5000)'),
        matchup('MCTS(n=5000)',  'MCTS(n=10000)'),
        matchup('MCTS(n=1000)',  'MCTS(n=10000)'),
    ]),

    # --- 6. Self-play (first-mover analysis) ---
    ('Self-play', [
        self_play('Alpha-Beta(d=4)'),
        self_play('Alpha-Beta(d=6)'),
        self_play('Alpha-Beta(d=8)'),
        self_play('MCTS(n=5000)'),
    ]),

    # --- 7. Random baseline ---
    ('Random baseline', [
        matchup('Alpha-Beta(d=4)', 'Random'),
    ]),

    # ============================================================
    # EXTENDED -- uncomment for deeper analysis (adds hours)
    # ============================================================

    # # --- High-N MCTS vs AB ---
    # ('MCTS high-N', [
    #     matchup('MCTS(n=20000)', 'Alpha-Beta(d=4)'),
    #     matchup('MCTS(n=20000)', 'Alpha-Beta(d=6)'),
    #     matchup('MCTS(n=50000)', 'Alpha-Beta(d=4)'),
    # ]),

    # # --- Rich eval function comparison ---
    # ('Eval: standard vs rich', [
    #     matchup('Alpha-Beta(d=4)', 'Alpha-Beta(d=4,rich)'),
    #     matchup('Alpha-Beta(d=6)', 'Alpha-Beta(d=6,rich)'),
    # ]),
]

# --- Run the tournament ---
total_matchups = sum(len(m) for _, m in scenarios)
total_games = total_matchups * 2 * GAMES_PER_SEAT
print(f'Tournament: {len(scenarios)} scenarios, {total_matchups} matchups, {total_games} games')
if CSV_PATH:
    print(f'CSV output: {CSV_PATH}')
    init_csv(CSV_PATH)
else:
    print('Output: screen only')
print()

import time as _time
overall_t0 = _time.time()
all_summaries = []

for sn, matchup_list in scenarios:
    summaries = run_scenario(matchup_list, n_games=GAMES_PER_SEAT,
                             csv_path=CSV_PATH, scenario_name=sn)
    all_summaries.extend(summaries)

elapsed = _time.time() - overall_t0
print(f'\nTournament complete: {elapsed:.0f}s ({elapsed/60:.1f} min)')
if CSV_PATH:
    print(f'Results saved to: {CSV_PATH}')

## Results Summary
Formatted table from the most recent tournament or matchup run.

In [ ]:
# Display results table
if all_summaries:
    print(f'{"Matchup":<45s} {"N":>4} {"P1W":>4} {"P2W":>4} {"D":>3} {"P1%":>5} {"AvgScore":>11} {"Mvs":>4}')
    print('-' * 82)
    for s in all_summaries:
        wr = s['p1_wins'] / s['total_games'] * 100
        print(f'{s["matchup_label"]:<45s} {s["total_games"]:>4} '
              f'{s["p1_wins"]:>4} {s["p2_wins"]:>4} {s["draws"]:>3} '
              f'{wr:>4.0f}% '
              f'{s["p1_avg_score"]:>5.1f}-{s["p2_avg_score"]:<5.1f} '
              f'{s["avg_moves"]:>4.0f}')
else:
    print('No results yet. Run a tournament (Cell 5) or matchup (Cell 4) first.')